# Lab 6 — Deep sequence models (RNN, LSTM, GRU)

**Goal:** forecast the next value from a **fixed-length window** of past \(y\) using **PyTorch** recurrent layers (**RNN**, **LSTM**, **GRU**), with scaling fit on the **train** segment only.

**Install:** `pip install torch` (CPU wheel is fine).

**Prerequisite:** `generate_data.ipynb`.

Notes: sequence models learn **nonlinear** mappings from lags; they still need careful **scaling**, **regularisation**, and **validation** — often **data-hungry** compared to linear / ARIMA.

## 1. Recurrent layers (idea)

At each time step \(t\), a cell updates a **hidden state** \(h_t\) from \(h_{t-1}\) and input \(x_t\). **LSTM/GRU** add gating to reduce vanishing-gradient issues vs plain **RNN**.

For univariate forecasting, \(x_t\) is often the scalar \(y_{t-1}\) (or an embedded window). Here we feed a **window** \((y_{t-L},\ldots,y_{t-1})\) as a sequence of length \(L\).

## 2. Pitfalls

- **Scale** — neural nets expect roughly O(1) inputs; use `MinMaxScaler` or standardisation fit on **train**.
- **Leakage** — do not fit scaler on test.
- **Horizons** — this notebook trains **one-step**; long horizons need iterative or multi-output heads.
- **Baselines** — compare to naive and linear models (Labs 3–5).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_PATH = Path("data/synthetic_series.csv")
META_PATH = Path("data/synthetic_series_meta.json")
df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
df = df.set_index("date").asfreq("D")
y = df["y"].astype(float)

TEST_H = 120
SEQ_LEN = 28
HIDDEN = 64
EPOCHS = 40
BATCH = 64
LR = 1e-3

split = len(y) - TEST_H
train_y, test_y = y.iloc[:split], y.iloc[split:]

scaler = MinMaxScaler()
scaler.fit(train_y.values.reshape(-1, 1))
all_s = scaler.transform(y.values.reshape(-1, 1)).flatten()


def make_windows(arr: np.ndarray, seq: int, i0: int, i1: int) -> tuple[np.ndarray, np.ndarray]:
    """Targets at indices i0..i1-1 use windows ending before them."""
    X, Y = [], []
    for i in range(i0, i1):
        X.append(arr[i - seq : i])
        Y.append(arr[i])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)


X_tr, Y_tr = make_windows(all_s, SEQ_LEN, SEQ_LEN, split)
X_te, Y_te = make_windows(all_s, SEQ_LEN, split, len(y))
print("Train windows:", X_tr.shape, "Test:", X_te.shape)

In [ ]:
class SeqNet(nn.Module):
    def __init__(self, kind: str, hidden: int = HIDDEN):
        super().__init__()
        if kind == "RNN":
            self.rnn = nn.RNN(1, hidden, batch_first=True)
        elif kind == "LSTM":
            self.rnn = nn.LSTM(1, hidden, batch_first=True)
        elif kind == "GRU":
            self.rnn = nn.GRU(1, hidden, batch_first=True)
        else:
            raise ValueError(kind)
        self.head = nn.Linear(hidden, 1)
        self._kind = kind

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, 1)
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)


def train_model(kind: str) -> tuple[SeqNet, np.ndarray]:
    device = torch.device("cpu")
    ds = TensorDataset(
        torch.from_numpy(X_tr).unsqueeze(-1),
        torch.from_numpy(Y_tr),
    )
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True)
    model = SeqNet(kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(EPOCHS):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        pred_te = model(torch.from_numpy(X_te).unsqueeze(-1)).numpy()
    return model, pred_te


results = {}
preds = {}
for k in ["RNN", "LSTM", "GRU"]:
    _, pred = train_model(k)
    preds[k] = pred
    inv = scaler.inverse_transform(pred.reshape(-1, 1)).flatten()
    yt = test_y.values
    rmse = float(np.sqrt(mean_squared_error(yt, inv)))
    mae = float(mean_absolute_error(yt, inv))
    results[k] = {"RMSE": rmse, "MAE": mae}
    print(f"{k:4s}  RMSE={rmse:.4f}  MAE={mae:.4f}")

pd.DataFrame(results).T

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test_y.index, test_y.values, "k", lw=1, label="Actual")
for k, c in zip(["RNN", "LSTM", "GRU"], ["C0", "C1", "C2"]):
    inv = scaler.inverse_transform(preds[k].reshape(-1, 1)).flatten()
    ax.plot(test_y.index, inv, lw=0.9, alpha=0.85, label=k, color=c)
ax.legend()
ax.set_title("Test: one-step-ahead (scaled train, inverse for plot)")
plt.tight_layout()
plt.show()

## Interview checklist

1. **Input window** vs **state** — RNN consumes a window; for streaming, you can carry hidden state.
2. **Vanishing gradients** — why LSTM/GRU exist.
3. **Sample efficiency** — neural nets often need more data than ARIMA / linear; **regularisation** (dropout, early stopping) matters.
4. **Uncertainty** — point forecasts here; quantiles need different heads or ensembles.

**Congratulations** — you have a full interview prep path from EDA to deep models.